# Example notebook

In [1]:
#reqs
import pandas as pd
import geopandas as gpd
import urllib
import json
import zzproxies
from zzproxies.core import registry
print("zzproxies version: ", zzproxies.__version__)

zzproxies version:  0.9.0


In [2]:
# helpers
def get_location_context(address: str, padding: float = 0.005):
    """
    Translates an address into a BBOX and an ISO Country Code.
    """
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(address)}&format=json&addressdetails=1&limit=1"
    headers = {"User-Agent": "zzproxies-research-lab"}
    
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
        
    if not data:
        raise ValueError(f"Address '{address}' not found.")
        
    loc = data[0]
    lat, lon = float(loc["lat"]), float(loc["lon"])
    country_code = loc.get("address", {}).get("country_code", "").upper()
    
    # Generate BBOX [xmin, ymin, xmax, ymax]
    bbox = [lon - padding, lat - padding, lon + padding, lat + padding]
    
    return bbox, country_code

import lonboard
from lonboard import SolidPolygonLayer, Map
import palettable
import numpy as np

def plot_building_choropleth(gdf, color_col='building_type'):
    # 1. Get unique building types and pick a palette
    unique_types = gdf[color_col].unique().tolist()
    # Tableau_10 is great for distinct categorical data
    palette = palettable.tableau.Tableau_10.colors 
    
    # 2. Create a lookup dictionary { 'Residential': [r, g, b], ... }
    color_lookup = {
        b_type: palette[i % len(palette)] 
        for i, b_type in enumerate(unique_types)
    }
    
    # 3. Apply colors to the GDF (Lonboard needs a list of [R, G, B] for each row)
    # We use .map() for speed and cast to a list of lists
    gdf['color'] = gdf[color_col].map(color_lookup)
    
    # 4. Handle any potential null colors (safety check)
    gdf['color'] = gdf['color'].apply(lambda x: x if x is not None else [200, 200, 200])

    # 5. Create the Layer
    layer = SolidPolygonLayer.from_geopandas(
        gdf,
        get_fill_color=np.array(gdf['color'].tolist(), dtype=np.uint8),
        opacity=0.8,
        pickable=True,
        auto_highlight=True
    )
    m = Map(layers=[layer])
    return m


In [3]:
# The research site
ADDRESS = "Otakaari 1, Espoo, Finland" 

# Geocode the intent
bbox, country_code = get_location_context(ADDRESS, padding=0.005)

print(f"📍 Research Site: {ADDRESS}")
print(f"🌍 Country Code: {country_code}")
print(f"📦 BBOX: {bbox}")

📍 Research Site: Otakaari 1, Espoo, Finland
🌍 Country Code: FI
📦 BBOX: [24.824994800000002, 60.1812949, 24.8349948, 60.1912949]


In [4]:
# Trigger the full pipeline

# Options: "upfront_gwp_supply", "activity_gwp_pcf", "slf_proxy"
PROXY_NAME = "biodiv_proxy" #"upfront_gwp_supply" #"activity_gwp_pcf"

results = None
try:
    results = registry.execute_pipeline(
        proxy_name=PROXY_NAME,
        bbox=bbox,
        country_code=country_code
    )
    
    metadata = registry.get_metadata(PROXY_NAME)
    status = metadata["status"]
    
    print(f"Execution Successful!")
    print(f"Proxy: {status.name} (v{status.version})")
    print(f"Methodology: {status.description}")
    print(f"---")
    print(f"✅ Processed {len(results)} buildings in the district.")

except PermissionError as e:
    print(f"❌ Geographic Error: {e}")
except Exception as e:
    print(f"❌ Pipeline Error: {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Execution Successful!
Proxy: Landcover Biodiversity Proxy (v1.0.0)
Methodology: Converts landcover composition into an ecological support score and GWPL label.
---
✅ Processed 330 buildings in the district.


In [7]:
#plot
df = pd.DataFrame(results)
gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.GeoSeries.from_wkt(df['wkt']), 
        crs="EPSG:4326"
    )
gdf = gdf[gdf.geometry.geom_type == 'Polygon']
m = plot_building_choropleth(gdf,color_col='GWPL')
m

In [9]:
#
print(df['GWPL'].value_counts())
df.head(3)


GWPL
L1: Strong Urban Habitat              175
L4: Critical Biodiversity Pressure    154
L0: High Biodiversity Support           1
Name: count, dtype: int64


,id,raw_overture_theme,raw_subtype,wkt,landcover_class,ecological_weight,area_sqm,ecological_area_sqm,landcover_composition,proxy_value,proxy_unit,biodiversity_index,GWPL
0,23b91a1d-e040-4c89-99df-0f1248497c84,building,medical,"POLYGON ((24.8316161 60.190824, 24.831731 60.1...",Building,0.0,1620.26,0.0,{'Building': 1620.26},0.0,ecological_area_sqm,0.0,L4: Critical Biodiversity Pressure
1,bc070836-d7a5-4f7e-af3b-801eac82535d,building,outbuilding,"POLYGON ((24.833834 60.1912622, 24.833841 60.1...",Building,0.0,61.85,0.0,{'Building': 61.85},0.0,ecological_area_sqm,0.0,L4: Critical Biodiversity Pressure
2,2c4148c4-0dcd-43df-b12a-84d497df2f3c,building,outbuilding,"POLYGON ((24.8334196 60.1909317, 24.8334471 60...",Building,0.0,36.81,0.0,{'Building': 36.81},0.0,ecological_area_sqm,0.0,L4: Critical Biodiversity Pressure


### Comparisons

In [ ]:
# -- comparison ---
from zzproxies import UrbanImpactReport, compare_reports

bbox_a, country_code = get_location_context("Mankkaa, Espoo", padding=0.003)
bbox_b, country_code = get_location_context("Malmi, Helsinki", padding=0.003)

# 1. Pipeline Execution
res_site_a = registry.execute_pipeline(PROXY_NAME, bbox_a, "FI")
res_site_b = registry.execute_pipeline(PROXY_NAME, bbox_b, "FI")

# 2. Report Creation
report_a = UrbanImpactReport("Mankkaa", res_site_a)
report_b = UrbanImpactReport("Malmi", res_site_b)

# 3. Comparison as DataFrame
comparison_table = pd.DataFrame(compare_reports([report_a, report_b]))

print("Impact-Verified Zoning Comparison ready!")
comparison_table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Impact-Verified Zoning Comparison ready!
